In [1]:
import cv2
import numpy as np
import os
import itertools
from tqdm import tqdm
import time
import sys
from scipy.fftpack import dct, idct

In [2]:
IMG_PATH = "/Users/valentindaveau/2IA_S8/Mission_R&D/Bee-recognition/images2_crop/M01C02_000066.png"
BASE_OUT = "/Users/valentindaveau/2IA_S8/Mission_R&D/Bee-recognition/Output"

img = cv2.imread(IMG_PATH, cv2.IMREAD_GRAYSCALE)
if img is None:
    raise FileNotFoundError(f"Image introuvable : {IMG_PATH}")
print(f"Image chargée — {img.shape[1]}x{img.shape[0]} px")

def make_output_dir(name):
    path = os.path.join(BASE_OUT, name)
    os.makedirs(path, exist_ok=True)
    return path

Image chargée — 1696x1031 px


## CLAHE — Contrast Limited Adaptive Histogram Equalization

Egalisation d'histogramme *locale* : l'image est divisee en tuiles, chaque tuile
recoit sa propre redistribution des niveaux de gris, puis les tuiles sont
recollees par interpolation bilineaire. La limite `clipLimit` empeche d'amplifier
le bruit dans les zones uniformes. Utile pour rehausser le contraste local
des abeilles face aux variations d'eclairage (gradient gauche/droite).

**Parametres :** `clip_limit` (force du contraste : 1=doux, 6=agressif),
`tile_size` (taille des tuiles en px — petite = correction tres locale).

In [ ]:
output_dir = make_output_dir("tests_clahe")

clip_limit_values = [1.0]
tile_size_values  = [32]
combinations = list(itertools.product(clip_limit_values, tile_size_values))
print(f"CLAHE — {len(combinations)} combinaisons.")

for clip, tile in tqdm(combinations, desc="CLAHE"):
    clahe  = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile))
    result = clahe.apply(img)
    cv2.imwrite(os.path.join(output_dir, f"test_clip{clip:.1f}_tile{tile}.png"), result)

print(f"Termine -> {output_dir}")

CLAHE — 20 combinaisons.


CLAHE: 100%|██████████| 20/20 [00:00<00:00, 42.30it/s]

Termine -> /Users/valentindaveau/2IA_S8/Mission_R&D/Bee-recognition/Output/tests_clahe


## Retinex — Multi-Scale Retinex (MSR)

Modele la perception humaine de la luminosite : separe la reflectance (texture,
abeilles) de l'illumination (gradient gauche/droite, ombres). Pour chaque echelle
sigma, calcule `log(image) - log(GaussianBlur(image, sigma))` — ce qui revient a
soustraire l'illumination basse frequence. Le MSR moyenne plusieurs echelles
pour couvrir a la fois les variations locales et globales.

**Parametres :** `sigmas` (echelles gaussiennes : petite=details locaux,
grande=correction globale de l'illumination).

In [ ]:
def apply_retinex(img, sigmas=(15, 80, 250)):
    """Multi-Scale Retinex sur image niveaux de gris."""
    img_f   = img.astype(np.float32) + 1.0
    log_img = np.log(img_f)
    msr     = np.zeros_like(img_f)
    for sigma in sigmas:
        blurred = cv2.GaussianBlur(img_f, (0, 0), sigma)
        blurred = np.maximum(blurred, 1.0)
        msr    += log_img - np.log(blurred)
    msr /= len(sigmas)
    msr  = cv2.normalize(msr, None, 0, 255, cv2.NORM_MINMAX)
    return msr.astype(np.uint8)


output_dir = make_output_dir("tests_retinex")

sigma_configs = {
    "ssr_15":    (15,),
    #"ssr_80":    (80,),
    #"ssr_250":   (250,),
    #"msr_std":   (15, 80, 250),
    #"msr_tight": (10, 25, 50),
    #"msr_wide":  (25, 100, 300),
}
print(f"Retinex — {len(sigma_configs)} configurations.")

for name, sigmas in tqdm(sigma_configs.items(), desc="Retinex"):
    result = apply_retinex(img, sigmas=sigmas)
    cv2.imwrite(os.path.join(output_dir, f"test_{name}.png"), result)

print(f"Termine -> {output_dir}")

Retinex — 6 configurations.


Retinex: 100%|██████████| 6/6 [00:02<00:00,  2.01it/s]

Termine -> /Users/valentindaveau/2IA_S8/Mission_R&D/Bee-recognition/Output/tests_retinex


## SNN — Symmetric Nearest Neighbour (filtre par plage d'intensite)

Pour chaque pixel c, on retient dans une fenetre de rayon `radius` uniquement
les voisins dont l'intensite est dans `[c - h, c + h]` (concept `query_radius`
du code SNN GitHub). La moyenne de ces voisins remplace c. Si aucun voisin
n'est retenu, c reste inchange.

Ce critere de selection par plage d'intensite est plus selectif que l'approche
paires spatiales symetriques : il preserve les bords forts (peu de voisins dans
la plage) et lisse les zones uniformes (beaucoup de voisins).

**Parametres :** `radius` (rayon spatial — garder <=2 pour les alveoles ~22 px),
`h` (rayon d'intensite — adapter au bruit : sigma~2.75 -> h~8-20),
`passes` (iterations successives).


In [4]:
# Conservee pour Kramer-Bruckner (center_weight >= 1)
def apply_snn_kb(img, radius, center_weight=0):
    """SNN paires symetriques (center_weight=0) ou KB pair-based (center_weight>=1)."""
    h, w = img.shape
    img_f = img.astype(np.float32)
    pad = cv2.copyMakeBorder(img_f, radius, radius, radius, radius, cv2.BORDER_REFLECT)

    offsets = [
        (dy, dx)
        for dy in range(-radius, radius + 1)
        for dx in range(-radius, radius + 1)
        if not (dy == 0 and dx == 0)
    ]
    pairs, seen = [], set()
    for dy, dx in offsets:
        if (-dy, -dx) not in seen:
            pairs.append(((dy, dx), (-dy, -dx)))
            seen.add((dy, dx))

    accum = img_f * float(center_weight)
    count = np.full_like(img_f, float(center_weight))
    for (dy1, dx1), (dy2, dx2) in pairs:
        a = pad[radius + dy1 : radius + dy1 + h, radius + dx1 : radius + dx1 + w]
        b = pad[radius + dy2 : radius + dy2 + h, radius + dx2 : radius + dx2 + w]
        closer = np.where(np.abs(a - img_f) <= np.abs(b - img_f), a, b)
        accum += closer
        count += 1.0
    return np.clip(accum / count, 0, 255).astype(np.uint8)


def apply_snn(img, radius, h=15, passes=1):
    """
    SNN par plage d'intensite : inspire du concept query_radius du code GitHub.
    Pour chaque pixel c, retient les voisins dans [c-h, c+h] et les moyenne.
    h : rayon d'intensite (sigma bruit ~2.75 -> h=8-20 conseille).
    """
    result = img.copy()
    for _ in range(passes):
        img_f = result.astype(np.float32)
        win   = 2 * radius + 1
        pad   = cv2.copyMakeBorder(img_f, radius, radius, radius, radius, cv2.BORDER_REFLECT)
        H, W  = img_f.shape

        # Empilement vectorise de tous les voisins : (n_neighbors, H, W)
        planes = [
            pad[dy:dy + H, dx:dx + W]
            for dy in range(win)
            for dx in range(win)
            if not (dy == radius and dx == radius)
        ]
        neighbors = np.stack(planes, axis=0)             # (n_nbrs, H, W)
        keep      = np.abs(neighbors - img_f) <= h       # masque booleen
        n_kept    = keep.sum(axis=0)                     # (H, W)
        accum     = (neighbors * keep).sum(axis=0)       # (H, W)

        # Aucun voisin retenu -> conserver le pixel original
        out    = np.where(n_kept > 0, accum / np.maximum(n_kept, 1), img_f)
        result = np.clip(out, 0, 255).astype(np.uint8)
    return result


In [ ]:
output_dir = make_output_dir("tests_snn")

# radius : rayon spatial (cellules ~22 px -> radius <= 2)
radius_values = [2]
# h : rayon d'intensite (bruit sigma~2.75 ; tester de 3*sigma a ~15*sigma)
h_values      = [5]
# passes : iterations successives
passes_values = [1,2,3]
combinations  = list(itertools.product(radius_values, h_values, passes_values))
print(f"SNN — {len(combinations)} combinaisons.")

for radius, h_val, passes in tqdm(combinations, desc="SNN"):
    result = apply_snn(img, radius, h=h_val, passes=passes)
    cv2.imwrite(os.path.join(output_dir, f"test_r{radius}_h{h_val}_passes{passes}.png"), result)

print(f"Termine -> {output_dir}")


SNN — 3 combinaisons.


SNN: 100%|██████████| 3/3 [00:03<00:00,  1.26s/it]

Termine -> /Users/valentindaveau/2IA_S8/Mission_R&D/Bee-recognition/Output/tests_snn


## EPOAF — Edge-Preserving Oriented Adaptive Filter

Lisse chaque pixel le long de la tangente locale au bord (direction perpendiculaire au gradient Sobel). Contrairement aux filtres isotropes, le lissage se fait *le long* des aretes sans les traverser. Utile pour uniformiser l'interieur des alveoles tout en conservant leurs bords nets.

**Parametres :** `offsets` (positions d'echantillonnage le long de la tangente), `ksize` (noyau Sobel : 3 = sensible aux petits bords, 5 = robuste au bruit), `passes`.

In [ ]:
def apply_epoaf(img, offsets, ksize=3):
    img_f = img.astype(np.float32)
    h, w = img.shape
    gx = cv2.Sobel(img_f, cv2.CV_32F, 1, 0, ksize=ksize)
    gy = cv2.Sobel(img_f, cv2.CV_32F, 0, 1, ksize=ksize)
    mag = np.sqrt(gx ** 2 + gy ** 2) + 1e-6
    tx = -gy / mag
    ty =  gx / mag
    xs = np.tile(np.arange(w, dtype=np.float32)[None, :], (h, 1))
    ys = np.tile(np.arange(h, dtype=np.float32)[:, None], (1, w))
    accum = img_f.copy()
    n = 1
    for s in offsets:
        map_x = np.clip(xs + s * tx, 0, w - 1).astype(np.float32)
        map_y = np.clip(ys + s * ty, 0, h - 1).astype(np.float32)
        accum += cv2.remap(img_f, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        n += 1
    return np.clip(accum / n, 0, 255).astype(np.uint8)


output_dir = make_output_dir("tests_epoaf")

# offsets : pas le long de la tangente a l'arete.
# Dans les zones plates (interieur alvéoles), le gradient Sobel est domine par le
# bruit -> direction tangente aleatoire. Un offset de 1px dans une direction
# aleatoire atterrit sur un pixel intérieur voisin (similaire) -> lissage bruit OK.
# Un offset >= 3px dans une direction aleatoire traverse les parois (~3-5px) -> flou.
# Regle : max_offset <= epaisseur_paroi / 2 ~ 1-2 px.
offset_configs = {
    "pm1":   (-1, 1),           # 3 px moyennes  : lissage minimal, bruit reduit
    #"pm1_2": (-2, -1, 1, 2),    # 5 px moyennes  : lissage modere, limite a 2px de la tangente
}

# ksize=3 : gradient local (reactif aux petits bords)
# ksize=5 : gradient plus lisse mais moins precis sur des parois de 3-5 px
ksize_values  = [5]

# passes : iterations ; avec pm1 le lissage est doux, 2-3 passes sont utiles
passes_values = [1]

combinations = [
    (off_name, off_vals, ks, passes)
    for off_name, off_vals in offset_configs.items()
    for ks in ksize_values
    for passes in passes_values
]
print(f"EPOAF — {len(combinations)} combinaisons.")

for off_name, off_vals, ks, passes in tqdm(combinations, desc="EPOAF"):
    result = img.copy()
    for _ in range(passes):
        result = apply_epoaf(result, off_vals, ksize=ks)
    cv2.imwrite(os.path.join(output_dir, f"test_{off_name}_ks{ks}_passes{passes}.png"), result)

print(f"Termine -> {output_dir}")


EPOAF — 12 combinaisons.


EPOAF: 100%|██████████| 12/12 [00:01<00:00,  6.20it/s]

Termine -> /Users/valentindaveau/2IA_S8/Mission_R&D/Bee-recognition/Output/tests_epoaf


## Bilateral

Filtre pondere par la proximite spatiale **et** la similarite d'intensite. Lisse les zones uniformes (interieur des alveoles, fond) tout en preservant les bords. Efficace sur le bruit gaussien mais plus lent que les filtres morphologiques.

**Parametres :** `d` (diametre de la fenetre), `sigmaColor` (tolerance d'intensite — faible = contours durs conserves), `sigmaSpace` (portee spatiale).

In [ ]:
output_dir = make_output_dir("tests_bilateral")

# d : diametre du filtre (cellules ~22 px -> rester en dessous pour ne pas traverser les bords)
d_values           = [110]
# sigmaColor : tolerance d'intensite (murs de cellules ~ 20-40 niveaux de contraste)
sigma_color_values = [20]
# sigmaSpace : sigma spatial (doit rester du meme ordre que d)
sigma_space_values = [50]
combinations = list(itertools.product(d_values, sigma_color_values, sigma_space_values))
print(f"Bilateral — {len(combinations)} combinaisons.")

for d, sc, ss in tqdm(combinations, desc="Bilateral"):
    filtered = cv2.bilateralFilter(img, d, sc, ss)
    cv2.imwrite(os.path.join(output_dir, f"test_d{d}_sc{sc}_ss{ss}.png"), filtered)

print(f"Termine -> {output_dir}")


## BM3D — Block-Matching 3D

Debruitage en deux passes. **Etape 1** : regroupe les blocs similaires dans l'image, applique un seuillage dur dans le domaine DCT 3D (hard-thresholding). **Etape 2** : affine par filtrage de Wiener collaboratif sur les memes groupes. Donne les meilleurs resultats sur le bruit gaussien mais est significativement plus lent que les autres filtres.

**Parametres :** `sigma` (niveau de bruit suppose), `Step1/2_ThreDist` (seuil de similarite des blocs), `Step1/2_WindowSize` (fenetre de recherche).

In [3]:
# ── Parametres BM3D ─────────────────────────────────────────────────────────
# sigma : bruit estime sur l image reelle (Immerkaer) = ~2.75 -> on prend 4
sigma              = 4
# lamb2d : seuillage pre-filtrage 2D (actif uniquement si sigma>40 -> jamais ici)
lamb2d             = 2.0
# lamb3d : facteur du seuil hard-thresholding DCT : seuil = lamb3d * sigma = 10.8
lamb3d             = 2.7
# Step1_ThreDist : distance max entre blocs DCT pour les considerer similaires
# reduit de 2500 a 1000 pour etre selectif (peu de bruit -> blocs similaires tres proches)
Step1_ThreDist     = 1000
Step1_MaxMatch     = 16
Step1_BlockSize    = 8
Step1_spdup_factor = 3
Step1_WindowSize   = 39
# Step2_ThreDist : reduit proportionnellement (400 -> 200)
Step2_ThreDist     = 200
Step2_MaxMatch     = 32
Step2_BlockSize    = 8
Step2_spdup_factor = 3
Step2_WindowSize   = 39
Kaiser_Window_beta = 2.0

# ── BM3D : fonctions internes ─────────────────────────────────────────────────

def Initialization(Img, BlockSize, Kaiser_Window_beta):
    InitImg    = np.zeros(Img.shape, dtype=float)
    InitWeight = np.zeros(Img.shape, dtype=float)
    w          = np.kaiser(BlockSize, Kaiser_Window_beta)
    InitKaiser = np.outer(w, w)
    return InitImg, InitWeight, InitKaiser


def SearchWindow(Img, RefPoint, BlockSize, WindowSize):
    if BlockSize >= WindowSize:
        raise ValueError("BlockSize must be smaller than WindowSize.")
    M = np.zeros((2, 2), dtype=int)
    M[0, 0] = max(0, RefPoint[0] + int((BlockSize - WindowSize) / 2))
    M[0, 1] = max(0, RefPoint[1] + int((BlockSize - WindowSize) / 2))
    M[1, 0] = M[0, 0] + WindowSize
    M[1, 1] = M[0, 1] + WindowSize
    if M[1, 0] >= Img.shape[0]:
        M[1, 0] = Img.shape[0] - 1
        M[0, 0] = M[1, 0] - WindowSize
    if M[1, 1] >= Img.shape[1]:
        M[1, 1] = Img.shape[1] - 1
        M[0, 1] = M[1, 1] - WindowSize
    return M


def dct2D(A):
    return dct(dct(A, axis=0, norm='ortho'), axis=1, norm='ortho')


def idct2D(A):
    return idct(idct(A, axis=0, norm='ortho'), axis=1, norm='ortho')


def PreDCT(Img, BlockSize):
    BlockDCT_all = np.zeros(
        (Img.shape[0] - BlockSize, Img.shape[1] - BlockSize, BlockSize, BlockSize), dtype=float
    )
    for i in range(BlockDCT_all.shape[0]):
        for j in range(BlockDCT_all.shape[1]):
            BlockDCT_all[i, j] = dct2D(Img[i:i+BlockSize, j:j+BlockSize].astype(np.float64))
    return BlockDCT_all


def Step1_ComputeDist(BlockDCT1, BlockDCT2):
    BlockSize = BlockDCT1.shape[0]
    if sigma > 40:
        thr = lamb2d * sigma
        BlockDCT1 = np.where(abs(BlockDCT1) < thr, 0, BlockDCT1)
        BlockDCT2 = np.where(abs(BlockDCT2) < thr, 0, BlockDCT2)
    return np.linalg.norm(BlockDCT1 - BlockDCT2) ** 2 / (BlockSize ** 2)


def Step1_Grouping(noisyImg, RefPoint, BlockDCT_all, BlockSize, ThreDist, MaxMatch, WindowSize):
    WL  = SearchWindow(noisyImg, RefPoint, BlockSize, WindowSize)
    N   = (WindowSize - BlockSize + 1) ** 2
    BlockPos   = np.zeros((N, 2), dtype=int)
    BlockGroup = np.zeros((N, BlockSize, BlockSize), dtype=float)
    Dist       = np.zeros(N, dtype=float)
    RefDCT     = BlockDCT_all[RefPoint[0], RefPoint[1]]
    cnt        = 0
    for i in range(WindowSize - BlockSize + 1):
        for j in range(WindowSize - BlockSize + 1):
            SearchedDCT = BlockDCT_all[WL[0, 0]+i, WL[0, 1]+j]
            d = Step1_ComputeDist(RefDCT, SearchedDCT)
            if d < ThreDist:
                BlockPos[cnt]   = [WL[0, 0]+i, WL[0, 1]+j]
                BlockGroup[cnt] = SearchedDCT
                Dist[cnt]       = d
                cnt += 1
    if cnt <= MaxMatch:
        return BlockPos[:cnt], BlockGroup[:cnt]
    idx = np.argpartition(Dist[:cnt], MaxMatch)
    return BlockPos[idx[:MaxMatch]], BlockGroup[idx[:MaxMatch]]


def Step1_3DFiltering(BlockGroup):
    thr = lamb3d * sigma
    nz  = 0
    for i in range(BlockGroup.shape[1]):
        for j in range(BlockGroup.shape[2]):
            v = dct(BlockGroup[:, i, j], norm='ortho')
            v[abs(v) < thr] = 0.0
            nz += np.nonzero(v)[0].size
            BlockGroup[:, i, j] = idct(v, norm='ortho')
    return BlockGroup, nz


def Step1_Aggregation(BlockGroup, BlockPos, basicImg, basicWeight, basicKaiser, nz):
    w = (1.0 if nz < 1 else 1.0 / (sigma ** 2 * nz)) * basicKaiser
    for i in range(BlockPos.shape[0]):
        r, c = BlockPos[i, 0], BlockPos[i, 1]
        s1, s2 = BlockGroup.shape[1], BlockGroup.shape[2]
        basicImg   [r:r+s1, c:c+s2] += w * idct2D(BlockGroup[i])
        basicWeight[r:r+s1, c:c+s2] += w


def BM3D_Step1(noisyImg):
    basicImg, basicWeight, basicKaiser = Initialization(noisyImg, Step1_BlockSize, Kaiser_Window_beta)
    BlockDCT_all = PreDCT(noisyImg, Step1_BlockSize)
    for i in range(int((noisyImg.shape[0] - Step1_BlockSize) / Step1_spdup_factor) + 2):
        for j in range(int((noisyImg.shape[1] - Step1_BlockSize) / Step1_spdup_factor) + 2):
            RefPoint = [
                min(Step1_spdup_factor * i, noisyImg.shape[0] - Step1_BlockSize - 1),
                min(Step1_spdup_factor * j, noisyImg.shape[1] - Step1_BlockSize - 1),
            ]
            BlockPos, BlockGroup = Step1_Grouping(
                noisyImg, RefPoint, BlockDCT_all,
                Step1_BlockSize, Step1_ThreDist, Step1_MaxMatch, Step1_WindowSize)
            BlockGroup, nz = Step1_3DFiltering(BlockGroup)
            Step1_Aggregation(BlockGroup, BlockPos, basicImg, basicWeight, basicKaiser, nz)
    basicImg /= np.where(basicWeight == 0, 1, basicWeight)
    return basicImg


def Step2_ComputeDist(img, P1, P2, BlockSize):
    B1 = img[P1[0]:P1[0]+BlockSize, P1[1]:P1[1]+BlockSize].astype(np.float64)
    B2 = img[P2[0]:P2[0]+BlockSize, P2[1]:P2[1]+BlockSize].astype(np.float64)
    return np.linalg.norm(B1 - B2) ** 2 / (BlockSize ** 2)


def Step2_Grouping(basicImg, noisyImg, RefPoint, BlockSize, ThreDist, MaxMatch, WindowSize,
                   BlockDCT_basic, BlockDCT_noisy):
    WL  = SearchWindow(basicImg, RefPoint, BlockSize, WindowSize)
    N   = (WindowSize - BlockSize + 1) ** 2
    BlockPos         = np.zeros((N, 2), dtype=int)
    BlockGroup_basic = np.zeros((N, BlockSize, BlockSize), dtype=float)
    BlockGroup_noisy = np.zeros((N, BlockSize, BlockSize), dtype=float)
    Dist             = np.zeros(N, dtype=float)
    cnt              = 0
    for i in range(WindowSize - BlockSize + 1):
        for j in range(WindowSize - BlockSize + 1):
            sp = [WL[0, 0]+i, WL[0, 1]+j]
            d  = Step2_ComputeDist(basicImg, RefPoint, sp, BlockSize)
            if d < ThreDist:
                BlockPos[cnt] = sp
                Dist[cnt]     = d
                cnt += 1
    if cnt <= MaxMatch:
        BlockPos = BlockPos[:cnt]
    else:
        BlockPos = BlockPos[np.argpartition(Dist[:cnt], MaxMatch)[:MaxMatch]]
    for i in range(BlockPos.shape[0]):
        r, c = BlockPos[i]
        BlockGroup_basic[i] = BlockDCT_basic[r, c]
        BlockGroup_noisy[i] = BlockDCT_noisy[r, c]
    n = BlockPos.shape[0]
    return BlockPos, BlockGroup_basic[:n], BlockGroup_noisy[:n]


def Step2_3DFiltering(BlockGroup_basic, BlockGroup_noisy):
    Weight = 0
    coef   = 1.0 / BlockGroup_noisy.shape[0]
    for i in range(BlockGroup_noisy.shape[1]):
        for j in range(BlockGroup_noisy.shape[2]):
            vb = dct(BlockGroup_basic[:, i, j], norm='ortho')
            vn = dct(BlockGroup_noisy[:, i, j], norm='ortho')
            vw = vb ** 2 * coef
            vw /= (vw + sigma ** 2)
            vn *= vw
            Weight += np.sum(vw)
            BlockGroup_noisy[:, i, j] = idct(vn, norm='ortho')
    WienerWeight = 1.0 / (sigma ** 2 * Weight) if Weight > 0 else 1.0
    return BlockGroup_noisy, WienerWeight


def Step2_Aggregation(BlockGroup_noisy, WienerWeight, BlockPos, finalImg, finalWeight, finalKaiser):
    w = WienerWeight * finalKaiser
    for i in range(BlockPos.shape[0]):
        r, c = BlockPos[i, 0], BlockPos[i, 1]
        s1, s2 = BlockGroup_noisy.shape[1], BlockGroup_noisy.shape[2]
        finalImg   [r:r+s1, c:c+s2] += w * idct2D(BlockGroup_noisy[i])
        finalWeight[r:r+s1, c:c+s2] += w


def BM3D_Step2(basicImg, noisyImg):
    finalImg, finalWeight, finalKaiser = Initialization(basicImg, Step2_BlockSize, Kaiser_Window_beta)
    BlockDCT_noisy = PreDCT(noisyImg, Step2_BlockSize)
    BlockDCT_basic = PreDCT(basicImg, Step2_BlockSize)
    for i in range(int((basicImg.shape[0] - Step2_BlockSize) / Step2_spdup_factor) + 2):
        for j in range(int((basicImg.shape[1] - Step2_BlockSize) / Step2_spdup_factor) + 2):
            RefPoint = [
                min(Step2_spdup_factor * i, basicImg.shape[0] - Step2_BlockSize - 1),
                min(Step2_spdup_factor * j, basicImg.shape[1] - Step2_BlockSize - 1),
            ]
            BlockPos, BG_basic, BG_noisy = Step2_Grouping(
                basicImg, noisyImg, RefPoint,
                Step2_BlockSize, Step2_ThreDist, Step2_MaxMatch, Step2_WindowSize,
                BlockDCT_basic, BlockDCT_noisy)
            BG_noisy, WW = Step2_3DFiltering(BG_basic, BG_noisy)
            Step2_Aggregation(BG_noisy, WW, BlockPos, finalImg, finalWeight, finalKaiser)
    finalImg /= np.where(finalWeight == 0, 1, finalWeight)
    return finalImg

In [4]:
output_dir = make_output_dir("tests_bm3d")
cv2.setUseOptimized(True)


img_f = img.astype(np.float64)

t0        = time.time()
basic_img = BM3D_Step1(img_f)
basic_u8  = np.zeros(img.shape)
cv2.normalize(basic_img, basic_u8, 0, 255, cv2.NORM_MINMAX, dtype=-1)
basic_u8   = basic_u8.astype(np.uint8)
basic_path = os.path.join(output_dir, "bm3d_custom_basic.png")
cv2.imwrite(basic_path, basic_u8)
t1 = time.time()
print(f"Basic estimate -> {basic_path}  ({t1 - t0:.1f}s)")

final_img  = BM3D_Step2(basic_img, img_f)
cv2.normalize(final_img, final_img, 0, 255, cv2.NORM_MINMAX, dtype=-1)
final_img  = final_img.astype(np.uint8)
final_path = os.path.join(output_dir, "bm3d_custom_final.png")
cv2.imwrite(final_path, final_img)
t2 = time.time()
print(f"Final estimate  -> {final_path}  ({t2 - t1:.1f}s)")

Basic estimate -> /Users/valentindaveau/2IA_S8/Mission_R&D/Bee-recognition/Output/tests_bm3d/bm3d_custom_basic.png  (496.2s)
Final estimate  -> /Users/valentindaveau/2IA_S8/Mission_R&D/Bee-recognition/Output/tests_bm3d/bm3d_custom_final.png  (628.2s)


## Kramer-Bruckner (morphologique)

Filtre non-lineaire iteratif base sur les extrema locaux. Pour chaque pixel, calcule le min et le max local (erosion/dilatation morphologique), puis choisit le min si le pixel est en dessous du point median local, le max sinon. Les iterations successives convergent vers une image "plate par morceaux" tout en renforcant les contours.

**Parametres :** `win` (demi-taille du voisinage), `ITERS` (iterations fixes), `MS_SCHEDULE` (sequence multi-echelle a fenetres croissantes).

In [11]:
def kb_filter(img_u8, win=1, shape="square"):
    img_f     = img_u8.astype(np.float32)
    k         = 2 * win + 1
    if shape == "ellipse":
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    else:
        kernel = np.ones((k, k), np.uint8)
    local_min = cv2.erode(img_f, kernel)
    local_max = cv2.dilate(img_f, kernel)
    midpoint  = (local_max + local_min) / 2.0
    out       = local_min.copy()
    out[img_f > midpoint] = local_max[img_f > midpoint]
    return np.clip(out, 0, 255).astype(np.uint8)


output_dir = make_output_dir("tests_kramer_bruckner")
PAD     = 4
LHEIGHT = 22

def labeled_tile(tile_img, text):
    bar = np.full((LHEIGHT, tile_img.shape[1]), 30, dtype=np.uint8)
    cv2.putText(bar, text, (4, LHEIGHT - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.42, 220, 1, cv2.LINE_AA)
    return np.vstack([bar, tile_img])

ITERS = 1

# ── Iterations carre ──────────────────────────────────────────────────────────
current = img.copy()
for i in range(ITERS):
    current = kb_filter(current, win=1, shape="square")
print(f"[square] {ITERS} iterations terminees")

cv2.imwrite(os.path.join(output_dir, f"kb_square_iter_{ITERS:02d}.png"), current)
print(f"[sortie] -> {output_dir}/kb_square_iter_{ITERS:02d}.png")

# ── Iterations ellipse (desactivees : resultat identique au carre sur cette image)
# current_el = img.copy()
# for i in range(ITERS):
#     current_el = kb_filter(current_el, win=1, shape="ellipse")
# print(f"[ellipse] {ITERS} iterations terminees")
# cv2.imwrite(os.path.join(output_dir, f"kb_ellipse_iter_{ITERS:02d}.png"), current_el)


# ── Grille de comparaison : original | resultat final ────────────────────────
cols   = [img, current]
labels = ["Original", f"KB carre iter {ITERS}"]
tiles  = [labeled_tile(c, l) for c, l in zip(cols, labels)]
div    = np.full((tiles[0].shape[0], PAD), 80, dtype=np.uint8)
grid   = np.hstack([tiles[0], div, tiles[1]])
cv2.imwrite(os.path.join(output_dir, "compare_grid.png"), grid)
print(f"[grille] -> {output_dir}/compare_grid.png")

# ── Grille comparaison carre vs ellipse (desactivee)
# cols2   = [img, current, current_el]
# labels2 = ["Original", f"Carre iter {ITERS}", f"Ellipse iter {ITERS}"]
# tiles2  = [labeled_tile(c, l) for c, l in zip(cols2, labels2)]
# grid2   = tiles2[0]
# for t in tiles2[1:]:
#     grid2 = np.hstack([grid2, div, t])
# cv2.imwrite(os.path.join(output_dir, "compare_square_vs_ellipse.png"), grid2)

# ── Difference absolue resultat final vs original ─────────────────────────────
diff     = cv2.absdiff(current, img)
diff_vis = cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX)
cv2.imwrite(os.path.join(output_dir, "diff_iter.png"), diff_vis)
print(f"[diff] max={int(diff.max())}  mean={diff.mean():.2f}")

# ── Differences carre vs ellipse (desactivees)
# diff_sq  = cv2.absdiff(current,    img)
# diff_el  = cv2.absdiff(current_el, img)
# diff_cmp = np.hstack([
#     labeled_tile(cv2.normalize(diff_sq, None, 0, 255, cv2.NORM_MINMAX), f"Diff Carre x{ITERS}"),
#     np.full((img.shape[0] + LHEIGHT, PAD), 80, dtype=np.uint8),
#     labeled_tile(cv2.normalize(diff_el, None, 0, 255, cv2.NORM_MINMAX), f"Diff Ellipse x{ITERS}"),
# ])
# cv2.imwrite(os.path.join(output_dir, "diff_square_vs_ellipse.png"), diff_cmp)

print(f"Termine -> {output_dir}")


[square] 1 iterations terminees
[sortie] -> /Users/valentindaveau/2IA_S8/Mission_R&D/Bee-recognition/Output/tests_kramer_bruckner/kb_square_iter_01.png
[grille] -> /Users/valentindaveau/2IA_S8/Mission_R&D/Bee-recognition/Output/tests_kramer_bruckner/compare_grid.png
[diff] max=82  mean=4.65
Termine -> /Users/valentindaveau/2IA_S8/Mission_R&D/Bee-recognition/Output/tests_kramer_bruckner


## NLM — Non-Local Means

Debruite chaque pixel en moyennant les pixels dont le **patch** local est similaire au sien, quelle que soit leur distance spatiale dans l'image. Tres efficace sur le bruit gaussien uniforme, mais sensible aux textures repetitives (risque de confondre des alveoles similaires si `h` est trop eleve).

**Parametres :** `h` (force de debruitage — compromis bruit/flou), `templateWindowSize` (taille du patch de comparaison), `searchWindowSize` (zone de recherche des patchs similaires).

In [13]:
output_dir = make_output_dir("tests_nlm")

# h : force du debruitage (bruit mesure sigma~2.75 -> h = 2-3x sigma, soit 5-10)
h_values        = [3, 5, 7]
# templateWindowSize : taille du patch de comparaison (impair, << taille cellule 22 px)
template_values = [5, 7,9,11]
# searchWindowSize : zone de recherche ; rester dans ~1-2 cellules
search_values   = [15, 21, 27]
combinations    = list(itertools.product(h_values, template_values, search_values))
print(f"NLM — {len(combinations)} combinaisons.")

for h, tmpl, srch in tqdm(combinations, desc="NLM"):
    filtered = cv2.fastNlMeansDenoising(img, h=h, templateWindowSize=tmpl, searchWindowSize=srch)
    cv2.imwrite(os.path.join(output_dir, f"test_h{h}_tmpl{tmpl}_srch{srch}.png"), filtered)

print(f"Termine -> {output_dir}")


NLM — 36 combinaisons.


NLM: 100%|██████████| 36/36 [00:09<00:00,  3.70it/s]

Termine -> /Users/valentindaveau/2IA_S8/Mission_R&D/Bee-recognition/Output/tests_nlm


---
## Pipeline complet — 18 combinaisons pour YOLO

Deux etapes enchainees sur chaque image source :

| Etape | Options |
|-------|---------|
| **Step 1 — contraste** | `clahe` · `original` · `retinex` |
| **Step 2 — lissage**   | `snn` · `bilateral` · `kb` · `epoaf` · `nlm` · `bm3d` |

**3 × 6 = 18 combinaisons** sauvegardees dans `Output/pipeline/` sous la forme `{source}_{filtre}.png`.

- `run_full_pipeline(img)` — execute les 18 combinaisons
- `run_combination(source, edge_filter, img)` — execute une seule combinaison

> **Note BM3D** : le BM3D custom est lent (~ 4-5 minutes sur 1696×1031 px).
> Desactiver avec `STEP2_FILTERS` si necessaire.

In [ ]:
# ── Parametres fixes du pipeline (meilleurs defaults par filtre) ─────────────
PIPELINE_PARAMS = {
    "clahe":    {"clip_limit": 2.0, "tile_size": 8},
    "retinex":  {"sigmas": (15, 80, 250)},
    "snn":      {"radius": 2, "h": 15, "passes": 1},
    "bilateral":{"d": 15, "sigmaColor": 20, "sigmaSpace": 15},
    "kb":       {"win": 1, "iters": 5},
    "epoaf":    {"offsets": (-2, -1, 1, 2), "ksize": 3},
    "nlm":      {"h": 7, "templateWindowSize": 7, "searchWindowSize": 21},
    "bm3d":     {},  # utilise les globaux sigma/lamb* definis dans la cellule BM3D
}

STEP1_SOURCES  = ["clahe", "original", "retinex"]
STEP2_FILTERS  = ["snn", "bilateral", "kb", "epoaf", "nlm", "bm3d"]


# ── Step 1 : contraste ───────────────────────────────────────────────────────
def apply_step1(img, source):
    if source == "clahe":
        p = PIPELINE_PARAMS["clahe"]
        clahe = cv2.createCLAHE(clipLimit=p["clip_limit"],
                                tileGridSize=(p["tile_size"], p["tile_size"]))
        return clahe.apply(img)
    if source == "retinex":
        return apply_retinex(img, sigmas=PIPELINE_PARAMS["retinex"]["sigmas"])
    return img.copy()  # original


# ── Step 2 : lissage / preservation des bords ────────────────────────────────
def apply_step2(img, edge_filter):
    p = PIPELINE_PARAMS[edge_filter]
    if edge_filter == "snn":
        return apply_snn(img, p["radius"], h=p["h"], passes=p["passes"])
    if edge_filter == "bilateral":
        return cv2.bilateralFilter(img, p["d"], p["sigmaColor"], p["sigmaSpace"])
    if edge_filter == "kb":
        result = img.copy()
        for _ in range(p["iters"]):
            result = kb_filter(result, win=p["win"])
        return result
    if edge_filter == "epoaf":
        return apply_epoaf(img, p["offsets"], ksize=p["ksize"])
    if edge_filter == "nlm":
        return cv2.fastNlMeansDenoising(
            img, h=p["h"],
            templateWindowSize=p["templateWindowSize"],
            searchWindowSize=p["searchWindowSize"],
        )
    if edge_filter == "bm3d":
        img_f  = img.astype(np.float64)
        basic  = BM3D_Step1(img_f)
        final  = BM3D_Step2(basic, img_f)
        out    = np.zeros(img.shape, dtype=np.float64)
        cv2.normalize(final, out, 0, 255, cv2.NORM_MINMAX, dtype=-1)
        return out.astype(np.uint8)
    raise ValueError(f"Filtre inconnu : {edge_filter}")


# ── Combinaison unique ───────────────────────────────────────────────────────
def run_combination(source, edge_filter, img, output_dir=None):
    """
    Execute une combinaison (source, edge_filter) et retourne l'image resultante.
    Si output_dir est fourni, sauvegarde {source}_{edge_filter}.png.

    source      : "clahe" | "original" | "retinex"
    edge_filter : "snn" | "bilateral" | "kb" | "epoaf" | "nlm" | "bm3d"
    """
    if source not in STEP1_SOURCES:
        raise ValueError(f"source invalide : {source}. Choisir parmi {STEP1_SOURCES}")
    if edge_filter not in STEP2_FILTERS:
        raise ValueError(f"filtre invalide : {edge_filter}. Choisir parmi {STEP2_FILTERS}")

    t0     = time.time()
    step1  = apply_step1(img, source)
    result = apply_step2(step1, edge_filter)
    elapsed = time.time() - t0

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        path = os.path.join(output_dir, f"{source}_{edge_filter}.png")
        cv2.imwrite(path, result)
        print(f"[{source:8s} + {edge_filter:9s}] sauvegarde -> {path}  ({elapsed:.1f}s)")
    return result


# ── Pipeline complet : 18 combinaisons ───────────────────────────────────────
def run_full_pipeline(img, output_dir=None):
    """
    Execute les 18 combinaisons (3 sources x 6 filtres).
    Retourne un dict {"source_filtre": img_array}.
    Si output_dir est fourni, sauvegarde toutes les images.
    """
    combinations = list(itertools.product(STEP1_SOURCES, STEP2_FILTERS))
    print(f"Pipeline — {len(combinations)} combinaisons (3 sources x 6 filtres)")
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    results = {}
    t_total = time.time()
    for source, edge_filter in tqdm(combinations, desc="Pipeline"):
        key = f"{source}_{edge_filter}"
        results[key] = run_combination(source, edge_filter, img, output_dir=output_dir)

    print(f"\nTermine — {len(results)} images en {time.time() - t_total:.1f}s")
    if output_dir:
        print(f"Sorties -> {output_dir}")
    return results


# ── Exemples d'utilisation ───────────────────────────────────────────────────
pipeline_out = make_output_dir("pipeline")

# -- Toutes les 18 combinaisons :
# results = run_full_pipeline(img, output_dir=pipeline_out)

# -- Une combinaison specifique :
# result = run_combination("clahe", "snn", img, output_dir=pipeline_out)
# result = run_combination("retinex", "nlm", img, output_dir=pipeline_out)
# result = run_combination("original", "bm3d", img, output_dir=pipeline_out)  # lent

# -- Toutes les combinaisons sauf BM3D (rapide) :
# STEP2_FILTERS_FAST = [f for f in STEP2_FILTERS if f != "bm3d"]
# for src, flt in itertools.product(STEP1_SOURCES, STEP2_FILTERS_FAST):
#     run_combination(src, flt, img, output_dir=pipeline_out)